# EDA y entrenamiento: ¿será un videojuego superventas?

**Dataset:** [Video Game Sales](https://www.kaggle.com/datasets/gregorut/videogamesales) de Kaggle (`vgsales.csv`): 16.598 videojuegos con más de 100.000 copias vendidas.

**Qué se hace en este notebook:**
1. Un **EDA pequeño** para entender los datos.
2. **Entrenar** un modelo (regresión logística) que predice si un juego será **superventas** (1 millón de copias o más) a partir de su plataforma, género, editora y año.
3. **Exportar** el modelo entrenado a `modelo_superventas.pkl`, que después usa `produccion.py` para simular un entorno de producción.

**Ejecución:** ejecuta la celda 0, reinicia el kernel si instaló algo y usa `Run All`.

## 0. Librerías

Comprueba que el kernel tiene las librerías necesarias y, si falta alguna, la instala en ese mismo entorno.

In [1]:
import importlib.util, subprocess, sys

paquetes = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "plotly": "plotly", "nbformat": "nbformat"}
faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltan])
    print("Instalado:", ", ".join(faltan), "→ reinicia el kernel (Restart) y ejecuta Run All.")
else:
    print("Todo instalado. Python:", sys.version.split()[0])

Instalado: plotly, nbformat → reinicia el kernel (Restart) y ejecuta Run All.


In [2]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "notebook_connected"
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#b4b8bf"


def grafico_simple(fig, titulo, eje_x, eje_y, altura=420):
    """Formato común: título que dice qué se mide y ejes con el nombre de la columna."""
    fig.update_layout(
        title={"text": titulo, "x": 0.5, "font": {"size": 20}},
        template="plotly_white", height=altura, font={"size": 14},
        showlegend=False, margin={"t": 70, "l": 10, "r": 40, "b": 60},
        xaxis_title=eje_x, yaxis_title=eje_y,
    )
    return fig

# 1. EDA (análisis exploratorio)

## 1.1 Un primer vistazo a los datos

In [3]:
df = pd.read_csv("vgsales.csv")
print(f"{df.shape[0]:,} juegos y {df.shape[1]} columnas")
display(df.head())

16,598 juegos y 11 columnas


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [4]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores distintos": df.nunique(),
    "nulos": df.isna().sum(),
})
display(resumen)

,tipo,valores distintos,nulos
Rank,int64,16598,0
Name,str,11493,0
Platform,str,31,0
Year,float64,39,271
Genre,str,12,0
Publisher,str,578,58
NA_Sales,float64,409,0
EU_Sales,float64,305,0
JP_Sales,float64,244,0
Other_Sales,float64,157,0


**Qué nos dice:**
- Cada fila es un juego **en una plataforma**. Las ventas están en **millones de unidades**.
- Solo hay nulos en `Year` (271 juegos) y `Publisher` (58). Son menos del 2 % y los imputará el modelo.
- `Publisher` tiene 578 editoras distintas: muchas publican muy pocos juegos.

## 1.2 La variable que vamos a predecir

Un juego es **superventas** si vende 1 millón de copias o más.

> **Datos usados:** columna `Global_Sales`. Se crea la columna `superventas` = 1 si `Global_Sales` ≥ 1 y 0 si no, y se **cuentan** los juegos de cada grupo.

In [5]:
df["superventas"] = (df["Global_Sales"] >= 1).astype(int)

tabla = df["superventas"].map({1: "Sí (≥ 1 millón)", 0: "No (< 1 millón)"}).value_counts().rename_axis("superventas").reset_index(name="numero_de_juegos")
tabla["porcentaje"] = (100 * tabla["numero_de_juegos"] / len(df)).round(1)
display(tabla)

fig = px.bar(tabla, x="superventas", y="numero_de_juegos", color="superventas",
             text=[f"{n:,} juegos ({p} %)" for n, p in zip(tabla["numero_de_juegos"], tabla["porcentaje"])],
             color_discrete_map={"Sí (≥ 1 millón)": AZUL, "No (< 1 millón)": GRIS})
fig.update_traces(textposition="outside", cliponaxis=False)
grafico_simple(fig, "Número de juegos superventas y no superventas",
               "superventas (Global_Sales ≥ 1)", "Número de juegos", 420).show()

,superventas,numero_de_juegos,porcentaje
0,No (< 1 millón),14517,87.5
1,Sí (≥ 1 millón),2081,12.5


**Conclusión:** solo 1 de cada 8 juegos es superventas. Las clases están **desbalanceadas**, y por eso no basta con mirar la *accuracy*: un modelo que dijera siempre "no" acertaría el 87 % sin servir para nada.

## 1.3 Superventas por género

> **Datos usados:** columnas `Genre` y `superventas`. Para cada género: número de juegos, cuántos son superventas y qué **porcentaje** representan.

In [6]:
tabla = (
    df.groupby("Genre")
    .agg(numero_de_juegos=("superventas", "size"), superventas=("superventas", "sum"))
    .assign(porcentaje_superventas=lambda t: (100 * t["superventas"] / t["numero_de_juegos"]).round(1))
    .sort_values("porcentaje_superventas", ascending=False)
    .reset_index()
)
display(tabla)

fig = px.bar(tabla, x="Genre", y="porcentaje_superventas", text="porcentaje_superventas")
fig.update_traces(marker_color=AZUL, texttemplate="%{text} %", textposition="outside", cliponaxis=False)
grafico_simple(fig, "% de juegos superventas en cada género",
               "Género (Genre)", "% de juegos superventas", 450).show()

,Genre,numero_de_juegos,superventas,porcentaje_superventas
0,Platform,886,195,22.0
1,Shooter,1310,252,19.2
2,Fighting,848,124,14.6
3,Racing,1249,182,14.6
4,Role-Playing,1488,203,13.6
5,Sports,2346,303,12.9
6,Action,3316,426,12.8
7,Simulation,867,93,10.7
8,Misc,1739,173,9.9
9,Puzzle,582,56,9.6


**Conclusión:** Platform (22 %) y Shooter (19 %) son los géneros con más superventas en proporción; Adventure (3 %) y Strategy (5 %), los que menos.

## 1.4 Superventas por editora

> **Datos usados:** columnas `Publisher` y `superventas`, solo para las **10 editoras que más juegos publican**. Para cada una: número de juegos y **porcentaje** de superventas.

In [7]:
top_editoras = df["Publisher"].value_counts().head(10).index
tabla = (
    df[df["Publisher"].isin(top_editoras)]
    .groupby("Publisher")
    .agg(numero_de_juegos=("superventas", "size"), superventas=("superventas", "sum"))
    .assign(porcentaje_superventas=lambda t: (100 * t["superventas"] / t["numero_de_juegos"]).round(1))
    .sort_values("porcentaje_superventas", ascending=False)
    .reset_index()
)
display(tabla)

fig = px.bar(tabla, x="porcentaje_superventas", y="Publisher", orientation="h", text="porcentaje_superventas")
fig.update_traces(marker_color=AZUL, texttemplate="%{text} %", textposition="outside", cliponaxis=False)
fig.update_yaxes(categoryorder="total ascending")
grafico_simple(fig, "% de juegos superventas de las 10 editoras con más juegos",
               "% de juegos superventas", "Editora (Publisher)", 460).show()

,Publisher,numero_de_juegos,superventas,porcentaje_superventas
0,Nintendo,703,339,48.2
1,Electronic Arts,1351,342,25.3
2,Take-Two Interactive,413,91,22.0
3,Sony Computer Entertainment,683,149,21.8
4,Activision,975,161,16.5
5,Ubisoft,921,117,12.7
6,THQ,715,88,12.3
7,Sega,639,74,11.6
8,Konami Digital Entertainment,832,62,7.5
9,Namco Bandai Games,932,53,5.7


**Conclusión:** el 48 % de los juegos de Nintendo son superventas, frente al 6 % de Namco Bandai o el 7 % de Konami. La editora es la variable que más diferencias muestra.

## 1.5 Juegos lanzados por año

> **Datos usados:** columna `Year`. Se **cuentan** los juegos de cada año (sin los 271 que no tienen año, y sin 2017 y 2020, que solo tienen 4 juegos).

In [8]:
tabla = (
    df.dropna(subset=["Year"]).query("Year <= 2016")
    .astype({"Year": int})
    .groupby("Year").size().reset_index(name="numero_de_juegos")
)
display(tabla.T)

fig = px.bar(tabla, x="Year", y="numero_de_juegos")
fig.update_traces(marker_color=AZUL)
grafico_simple(fig, "Número de juegos lanzados cada año",
               "Año de lanzamiento (Year)", "Número de juegos", 400).show()

,0,1,2,3,4,5,6,7,8,9,...,27,28,29,30,31,32,33,34,35,36
Year,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,...,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016
numero_de_juegos,9,46,36,17,14,14,21,16,15,17,...,1202,1428,1431,1259,1139,657,546,582,614,344


**Conclusión:** más de la mitad de los juegos del dataset salió entre 2005 y 2011 (la época de PS2, Wii, DS y Xbox 360); el máximo es 2009.

## 1.6 Conclusiones del EDA

- Solo el **12,5 %** de los juegos es superventas: las clases están desbalanceadas.
- **Género**, **editora**, **plataforma** y **año** muestran diferencias claras, así que serán las variables del modelo.
- **No** se usan las ventas por región (`NA_Sales`, `EU_Sales`...): no se conocen antes de lanzar el juego y su suma es justo lo que queremos predecir (sería hacer trampa, *data leakage*).

# 2. Entrenamiento del modelo

## 2.1 Preparar los datos

In [9]:
from sklearn.model_selection import train_test_split

VARIABLES = ["Year", "Platform", "Genre", "Publisher"]
X = df[VARIABLES]
y = df["superventas"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Entrenamiento: {len(X_train):,} juegos · Prueba: {len(X_test):,} juegos")

ModuleNotFoundError: No module named 'sklearn.utils.fixes'

## 2.2 Pipeline y entrenamiento

Todo el preprocesamiento va dentro del `Pipeline`, así el `.pkl` incluye también la preparación de los datos y en producción basta con pasarle los datos "en crudo".

- `Year`: se rellenan los nulos con la mediana y se escala.
- `Platform`, `Genre`, `Publisher`: se rellenan los nulos con el valor más frecuente y se aplica One-Hot Encoding. Las editoras o plataformas con menos de 30 juegos se agrupan en una sola columna (`min_frequency=30`), y si en producción llega una que el modelo no conoce, se trata igual (`handle_unknown="infrequent_if_exist"`).
- `LogisticRegression(class_weight="balanced")`: da más peso a los superventas para compensar que sean pocos.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocesador = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), ["Year"]),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)),
    ]), ["Platform", "Genre", "Publisher"]),
])

modelo = Pipeline([
    ("preprocesador", preprocesador),
    ("clasificador", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
modelo.fit(X_train, y_train)
print("Modelo entrenado.")

Modelo entrenado.


## 2.3 ¿Qué tal funciona?

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, recall_score, precision_score

y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]
(tn, fp), (fn, tp) = confusion_matrix(y_test, y_pred)

print(f"✔ Detecta {recall_score(y_test, y_pred):.0%} de los superventas ({tp} de {tp + fn})")
print(f"✔ De los juegos que marca como superventas, acierta el {precision_score(y_test, y_pred):.0%}"
      f" (elegir al azar acertaría el {y_test.mean():.0%})")
print(f"✔ AUC: {roc_auc_score(y_test, y_prob):.2f}  (0,5 = azar · 1 = perfecto)")

✔ Detecta 73% de los superventas (302 de 416)
✔ De los juegos que marca como superventas, acierta el 28% (elegir al azar acertaría el 13%)
✔ AUC: 0.81  (0,5 = azar · 1 = perfecto)


> **Datos usados:** los 3.320 juegos del conjunto de **prueba** (que el modelo no vio al entrenar). Para cada uno se compara lo que **predice** el modelo con el valor **real** de `superventas`, y se cuentan los 4 casos posibles.

In [ ]:
tabla = pd.DataFrame({
    "caso": ["Superventas detectado", "Superventas que se le escapa", "No superventas bien descartado", "Falsa alarma"],
    "real": ["Sí", "Sí", "No", "No"],
    "predicho": ["Sí", "No", "No", "Sí"],
    "numero_de_juegos": [tp, fn, tn, fp],
    "acierto": ["Acierto", "Fallo", "Acierto", "Fallo"],
})
display(tabla)

fig = px.bar(tabla, x="caso", y="numero_de_juegos", color="acierto", text="numero_de_juegos",
             color_discrete_map={"Acierto": AZUL, "Fallo": NARANJA})
fig.update_traces(textposition="outside", cliponaxis=False)
grafico_simple(fig, "Predicciones del modelo en los juegos de prueba (azul = acierto, naranja = fallo)",
               "Caso (real frente a predicho)", "Número de juegos", 450).show()

,caso,real,predicho,numero_de_juegos,acierto
0,Superventas detectado,Sí,Sí,302,Acierto
1,Superventas que se le escapa,Sí,No,114,Fallo
2,No superventas bien descartado,No,No,2109,Acierto
3,Falsa alarma,No,Sí,795,Fallo


**Conclusión:** de los 416 superventas reales detecta 302 y se le escapan 114. A cambio da bastantes **falsas alarmas**: el modelo prefiere avisar de más que perderse un éxito.

## 2.4 Probar el modelo con varios juegos

> **Datos usados:** 6 juegos inventados que se le pasan al modelo, con las mismas 4 columnas que usó para entrenar: `Year`, `Platform`, `Genre` y `Publisher`. El modelo devuelve la **probabilidad** de que cada uno sea superventas.

In [ ]:
ejemplos = pd.DataFrame({
    "Year": [2016, 2016, 2016, 2016, 2016, 2016],
    "Platform": ["PS4", "WiiU", "XOne", "PS4", "PC", "3DS"],
    "Genre": ["Shooter", "Platform", "Sports", "Shooter", "Adventure", "Puzzle"],
    "Publisher": ["Activision", "Nintendo", "Electronic Arts", "Ubisoft", "Unknown", "Unknown"],
})
ejemplos["probabilidad_superventas_%"] = (100 * modelo.predict_proba(ejemplos)[:, 1]).round(1)
ejemplos["juego"] = ejemplos["Genre"] + " de " + ejemplos["Publisher"] + " (" + ejemplos["Platform"] + ")"
display(ejemplos)

fig = px.bar(ejemplos, x="probabilidad_superventas_%", y="juego", orientation="h", text="probabilidad_superventas_%")
fig.update_traces(marker_color=AZUL, texttemplate="%{text} %", textposition="outside", cliponaxis=False)
fig.update_yaxes(categoryorder="total ascending")
fig.add_vline(x=50, line_dash="dash", line_color="#52514e", annotation_text="50 %: a partir de aquí dice 'superventas'")
fig.update_xaxes(range=[0, 105])
grafico_simple(fig, "Probabilidad de ser superventas que da el modelo a 6 juegos",
               "Probabilidad de superventas (%)", "Juego (Genre de Publisher, Platform)", 420).show()

,Year,Platform,Genre,Publisher,probabilidad_superventas_%,juego
0,2016,PS4,Shooter,Activision,79.5,Shooter de Activision (PS4)
1,2016,WiiU,Platform,Nintendo,73.8,Platform de Nintendo (WiiU)
2,2016,XOne,Sports,Electronic Arts,65.2,Sports de Electronic Arts (XOne)
3,2016,PS4,Shooter,Ubisoft,73.0,Shooter de Ubisoft (PS4)
4,2016,PC,Adventure,Unknown,0.6,Adventure de Unknown (PC)
5,2016,3DS,Puzzle,Unknown,1.4,Puzzle de Unknown (3DS)


**Conclusión:** el modelo da probabilidades altas a juegos de editoras grandes y géneros de éxito (un shooter de Activision o un plataformas de Nintendo), y muy bajas a juegos de editoras desconocidas en géneros con pocos superventas.

# 3. Exportar el modelo a `.pkl`

Se guarda el pipeline entrenado con `pickle`. Junto al modelo se guardan las variables que espera, sus métricas y la versión de scikit-learn, para que `produccion.py` pueda comprobar que todo cuadra.

In [ ]:
import datetime
import sklearn

artefacto = {
    "modelo": modelo,
    "variables": VARIABLES,
    "umbral": 0.5,
    "metricas": {
        "recall": round(recall_score(y_test, y_pred), 3),
        "precision": round(precision_score(y_test, y_pred), 3),
        "auc": round(roc_auc_score(y_test, y_prob), 3),
    },
    "sklearn_version": sklearn.__version__,
    "fecha_entrenamiento": datetime.datetime.now().isoformat(timespec="seconds"),
}

ruta_pkl = Path("modelo_superventas.pkl")
with open(ruta_pkl, "wb") as archivo:
    pickle.dump(artefacto, archivo)
print(f"Modelo exportado: {ruta_pkl.resolve()} ({ruta_pkl.stat().st_size / 1024:.0f} KB)")

Modelo exportado: C:\Users\pablo\OneDrive\Escritorio\modelo_superventas\modelo_superventas.pkl (20 KB)


Comprobación: se vuelve a cargar el `.pkl` y se predice un juego.

In [ ]:
with open(ruta_pkl, "rb") as archivo:
    cargado = pickle.load(archivo)

juego = pd.DataFrame([{"Year": 2016, "Platform": "PS4", "Genre": "Shooter", "Publisher": "Activision"}])
probabilidad = cargado["modelo"].predict_proba(juego)[0, 1]
print(f"Shooter de Activision para PS4 (2016) → {probabilidad:.0%} de probabilidad de ser superventas")
print("Ahora ejecuta en la terminal:  python produccion.py")

Shooter de Activision para PS4 (2016) → 79% de probabilidad de ser superventas
Ahora ejecuta en la terminal:  python produccion.py
